# Validation: Summer 2026 vs Legacy LUVit Spreadsheet

Compares two forecasts on households and employment.
End year (2050) only.

| Forecast | Source |
|---|---|
| **Summer 2026** | `examples/summer_2026/data/pipeline.h5` |
| **Legacy LUVit (RTP26)** | `examples/legacy_luvit_spreadsheet/output/Control-Totals-RTP26-2025-06-11.xlsx` |


In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Make the control_totals package importable from the notebook's location.
# This notebook lives at examples/summer_2026/, so .parents[1] is the repo root.
_REPO_ROOT = Path(".").resolve().parents[1]
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

from control_totals.util.pipeline import Pipeline  # noqa: E402

# ── Paths ──────────────────────────────────────────────────────────────────
_HERE = Path(".").resolve()                              # examples/summer_2026
_SUMMER_2026_CONFIGS = _HERE / "configs"
_LEGACY_WORKBOOK = (
    _HERE.parent / "legacy_luvit_spreadsheet" / "output"
    / "Control-Totals-RTP26-2025-06-11.xlsx"
)


In [ ]:
# ── Load Summer 2026 from pipeline.h5 ──────────────────────────────────────
_pipeline = Pipeline(settings_path=str(_SUMMER_2026_CONFIGS))

s26_hh  = _pipeline.get_table("rebased_control_totals_hh")
s26_emp = _pipeline.get_table("rebased_control_totals_emp")
s26_xwalk = (
    _pipeline.get_table("control_target_xwalk")
    [["target_id", "target_name", "control_id", "county_id"]]
    .drop_duplicates()
)

# ── Load Legacy LUVit from Excel ────────────────────────────────────────────
leg_hh  = pd.read_excel(_LEGACY_WORKBOOK, sheet_name="HH")
leg_emp = pd.read_excel(_LEGACY_WORKBOOK, sheet_name="Emp")
# Reuse the Summer 2026 crosswalk (same control area geography).
leg_xwalk = s26_xwalk.copy()

# Coerce year column labels to strings for consistent handling.
def _str_year_cols(df):
    return df.rename(columns={c: str(c) for c in df.columns if str(c).isdigit() and len(str(c)) == 4})

s26_hh  = _str_year_cols(s26_hh)
s26_emp = _str_year_cols(s26_emp)
leg_hh  = _str_year_cols(leg_hh)
leg_emp = _str_year_cols(leg_emp)


In [ ]:
_COUNTIES = [(53033, "King"), (53053, "Pierce"), (53061, "Snohomish"), (53035, "Kitsap")]
_COUNTY_IDS = [c for c, _ in _COUNTIES]
_COUNTY_NAMES = {c: n for c, n in _COUNTIES}


def _aggregate_county(df, xwalk, end_year):
    """Aggregate *df* to a county Series indexed by county name."""
    merged = df[["control_id", end_year]].merge(
        xwalk[["control_id", "county_id"]], on="control_id", how="inner"
    )
    grouped = merged.groupby("county_id", as_index=True)[end_year].sum()
    grouped = grouped.reindex(_COUNTY_IDS)
    grouped.index = [_COUNTY_NAMES[c] for c in _COUNTY_IDS]
    grouped.index.name = "County"
    grouped.loc["Region"] = grouped.sum()
    return grouped


def comparison_table(df_a, xwalk_a, df_b, xwalk_b, end_year, label_a, label_b):
    """Build a combined county comparison table with diff and % diff columns."""
    s_a = _aggregate_county(df_a, xwalk_a, end_year)
    s_b = _aggregate_county(df_b, xwalk_b, end_year)
    result = pd.DataFrame({label_a: s_a, label_b: s_b})
    result["Diff"] = result[label_a] - result[label_b]
    result["% Diff"] = result["Diff"] / result[label_b].replace(0, float("nan")) * 100
    fmt = {label_a: "{:,.0f}", label_b: "{:,.0f}", "Diff": "{:,.0f}", "% Diff": "{:.1f}%"}
    return result.style.format(fmt)


def target_comparison_table(df_a, xwalk_a, df_b, xwalk_b, end_year, county_id, label_a, label_b):
    """Build a target-area comparison table for a single county.

    Rows are target_names within the county, sorted by target_id, with a Total row appended.
    Columns: label_a, label_b, Diff, % Diff.
    """
    full_xwalk = xwalk_a[["target_id", "target_name", "control_id", "county_id"]].drop_duplicates()

    def _agg(df, xwalk):
        cols = ["control_id", end_year]
        merged = df[cols].merge(
            xwalk[["target_id", "target_name", "control_id", "county_id"]].drop_duplicates(),
            on="control_id", how="inner"
        )
        merged = merged[merged["county_id"] == county_id]
        grouped = (
            merged.groupby(["target_id", "target_name"], as_index=False)[end_year]
            .sum()
            .sort_values("target_id")
            .set_index("target_name")
            .drop(columns="target_id")
        )
        grouped.index.name = "Target Area"
        return grouped[end_year]

    s_a = _agg(df_a, xwalk_a)
    s_b = _agg(df_b, xwalk_b)

    result = pd.DataFrame({label_a: s_a, label_b: s_b})
    result.loc["Total"] = result.sum()
    result["Diff"] = result[label_a] - result[label_b]
    result["% Diff"] = result["Diff"] / result[label_b].replace(0, float("nan")) * 100
    fmt = {label_a: "{:,.0f}", label_b: "{:,.0f}", "Diff": "{:,.0f}", "% Diff": "{:.1f}%"}
    return result.style.format(fmt)


## Households


In [ ]:
display(comparison_table(s26_hh, s26_xwalk, leg_hh, leg_xwalk, "2050",
                         label_a="Summer 2026", label_b="Legacy LUVit (RTP26)"))


## Employment


In [ ]:
display(comparison_table(s26_emp, s26_xwalk, leg_emp, leg_xwalk, "2050",
                         label_a="Summer 2026", label_b="Legacy LUVit (RTP26)"))


---

## Households by Target Area


### King County


In [ ]:
display(target_comparison_table(s26_hh, s26_xwalk, leg_hh, leg_xwalk, "2050", 53033,
                                label_a="Summer 2026", label_b="Legacy LUVit (RTP26)"))


### Pierce County


In [ ]:
display(target_comparison_table(s26_hh, s26_xwalk, leg_hh, leg_xwalk, "2050", 53053,
                                label_a="Summer 2026", label_b="Legacy LUVit (RTP26)"))


### Snohomish County


In [ ]:
display(target_comparison_table(s26_hh, s26_xwalk, leg_hh, leg_xwalk, "2050", 53061,
                                label_a="Summer 2026", label_b="Legacy LUVit (RTP26)"))


### Kitsap County


In [ ]:
display(target_comparison_table(s26_hh, s26_xwalk, leg_hh, leg_xwalk, "2050", 53035,
                                label_a="Summer 2026", label_b="Legacy LUVit (RTP26)"))


---

## Employment by Target Area


### King County


In [ ]:
display(target_comparison_table(s26_emp, s26_xwalk, leg_emp, leg_xwalk, "2050", 53033,
                                label_a="Summer 2026", label_b="Legacy LUVit (RTP26)"))


### Pierce County


In [ ]:
display(target_comparison_table(s26_emp, s26_xwalk, leg_emp, leg_xwalk, "2050", 53053,
                                label_a="Summer 2026", label_b="Legacy LUVit (RTP26)"))


### Snohomish County


In [ ]:
display(target_comparison_table(s26_emp, s26_xwalk, leg_emp, leg_xwalk, "2050", 53061,
                                label_a="Summer 2026", label_b="Legacy LUVit (RTP26)"))


### Kitsap County


In [ ]:
display(target_comparison_table(s26_emp, s26_xwalk, leg_emp, leg_xwalk, "2050", 53035,
                                label_a="Summer 2026", label_b="Legacy LUVit (RTP26)"))
